# English Learning Analytics - Data Analysis

**Goal:** Explore processed data to find insights about learning patterns

**6 Key Queries:**
1. Error rate by level
2. Hardest words
3. Learning curve (attempt vs accuracy)
4. Time vs accuracy correlation
5. User performance distribution
6. **Retention analysis (for Sweet Spot chart)**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load processed data
df = pd.read_csv('../processed/quiz_results_clean.csv', sep=';', encoding='utf-8-sig')

print(f"Loaded {len(df):,} quiz results")
print(f"Columns: {df.columns.tolist()}")
df.head()

Loaded 10,000 quiz results
Columns: ['result_id', 'user_id', 'user_level', 'word_id', 'word_level', 'frequency', 'timestamp', 'is_correct', 'time_spent', 'attempt_number', 'days_since_last_seen', 'level_gap', 'is_hard_word', 'is_slow_response', 'is_rare_word', 'forgetting_risk', 'user_performance_score']


,result_id,user_id,user_level,word_id,word_level,frequency,timestamp,is_correct,time_spent,attempt_number,days_since_last_seen,level_gap,is_hard_word,is_slow_response,is_rare_word,forgetting_risk,user_performance_score
0,R01390,U001,B2,W0976,B2,high,2024-01-02,True,3.72,1,0,0,0,0,0,0,1.00
1,R01035,U001,B2,W1655,A2,high,2024-01-03,True,4.70,1,0,-2,0,0,0,0,1.00
2,R03109,U001,B2,W1655,A2,high,2024-01-04,True,5.27,2,1,-2,0,0,0,0,1.00
3,R06587,U001,B2,W0382,C2,low,2024-01-05,False,5.69,1,0,2,1,0,1,0,0.75
4,R07302,U001,B2,W1575,C1,low,2024-01-07,True,6.02,1,0,1,1,0,1,0,0.80


---
## Query 1: Error Rate by Level

**Question:** How does accuracy vary by word difficulty level?

In [5]:
# Calculate error rate by word level
error_by_level = df.groupby('word_level').agg({
    'is_correct': ['mean', 'count']
}).round(3)

error_by_level.columns = ['accuracy', 'count']
error_by_level['error_rate'] = 1 - error_by_level['accuracy']

# Order by CEFR level
level_order = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
error_by_level = error_by_level.reindex(level_order)

print("ERROR RATE BY WORD LEVEL")
print("=" * 50)
print(error_by_level)
print("\n📊 Insight:")
print(f"  - Lowest error: {error_by_level['error_rate'].idxmin()} ({error_by_level['error_rate'].min():.1%})")
print(f"  - Highest error: {error_by_level['error_rate'].idxmax()} ({error_by_level['error_rate'].max():.1%})")

ERROR RATE BY WORD LEVEL
            accuracy  count  error_rate
word_level                             
A1             0.790   1655       0.210
A2             0.726   1600       0.274
B1             0.621   1559       0.379
B2             0.547   1856       0.453
C1             0.452   1570       0.548
C2             0.341   1760       0.659

📊 Insight:
  - Lowest error: A1 (21.0%)
  - Highest error: C2 (65.9%)


---
## Query 2: Hardest Words

**Question:** Which words are most frequently answered incorrectly?

In [6]:
# Calculate error rate per word (min 10 attempts)
word_stats = df.groupby('word_id').agg({
    'is_correct': ['mean', 'count'],
    'word_level': 'first',
    'frequency': 'first'
})

word_stats.columns = ['accuracy', 'attempts', 'level', 'frequency']
word_stats['error_rate'] = 1 - word_stats['accuracy']

# Filter: at least 10 attempts
word_stats_filtered = word_stats[word_stats['attempts'] >= 10]

# Top 10 hardest words
hardest_words = word_stats_filtered.nlargest(10, 'error_rate')[[
    'error_rate', 'accuracy', 'attempts', 'level', 'frequency'
]]

print("TOP 10 HARDEST WORDS (min 10 attempts)")
print("=" * 70)
print(hardest_words.to_string())
print("\n📊 Insight:")
print(f"  - Average error rate of top 10: {hardest_words['error_rate'].mean():.1%}")
print(f"  - Most common level: {hardest_words['level'].mode().values[0]}")
print(f"  - Most common frequency: {hardest_words['frequency'].mode().values[0]}")

TOP 10 HARDEST WORDS (min 10 attempts)
         error_rate  accuracy  attempts level frequency
word_id                                                
W1999      1.000000  0.000000        10    C2       low
W1802      0.916667  0.083333        12    C1       low
W0031      0.900000  0.100000        10    C2       low
W1197      0.900000  0.100000        10    C2       low
W0060      0.866667  0.133333        15    B2    medium
W1967      0.833333  0.166667        12    C2      high
W1785      0.818182  0.181818        11    C2      high
W1949      0.818182  0.181818        11    C2      high
W0158      0.800000  0.200000        10    B2       low
W0490      0.800000  0.200000        10    C2    medium

📊 Insight:
  - Average error rate of top 10: 86.5%
  - Most common level: C2
  - Most common frequency: low


---
## Query 3: Learning Curve

**Question:** Does accuracy improve with more attempts?

In [7]:
# Accuracy by attempt number (limit to first 10 attempts)
learning_curve = df[df['attempt_number'] <= 10].groupby('attempt_number').agg({
    'is_correct': ['mean', 'count']
})

learning_curve.columns = ['accuracy', 'count']

print("LEARNING CURVE (First 10 Attempts)")
print("=" * 50)
print(learning_curve.to_string())
print("\n📊 Insight:")
improvement = learning_curve.loc[10, 'accuracy'] - learning_curve.loc[1, 'accuracy']
print(f"  - Accuracy at attempt 1: {learning_curve.loc[1, 'accuracy']:.1%}")
if 10 in learning_curve.index:
    print(f"  - Accuracy at attempt 10: {learning_curve.loc[10, 'accuracy']:.1%}")
    print(f"  - Improvement: {improvement:+.1%}")

LEARNING CURVE (First 10 Attempts)
                accuracy  count
attempt_number                 
1               0.576322   6014
2               0.607052   2354
3               0.552083    960
4               0.508772    399
5               0.436709    158
6               0.521127     71
7               0.592593     27
8               0.636364     11
9               0.800000      5
10              1.000000      1

📊 Insight:
  - Accuracy at attempt 1: 57.6%
  - Accuracy at attempt 10: 100.0%
  - Improvement: +42.4%


---
## Query 4: Time vs Accuracy Correlation

**Question:** Do users who spend more time answer more accurately?

In [8]:
# Create time buckets
df['time_bucket'] = pd.cut(df['time_spent'], 
                            bins=[0, 5, 10, 15, 300],
                            labels=['0-5s', '5-10s', '10-15s', '15s+'])

time_accuracy = df.groupby('time_bucket', observed=True).agg({
    'is_correct': ['mean', 'count']
})

time_accuracy.columns = ['accuracy', 'count']

print("ACCURACY BY TIME SPENT")
print("=" * 50)
print(time_accuracy.to_string())

# Calculate correlation
correlation = df['time_spent'].corr(df['is_correct'])

print("\n📊 Insight:")
print(f"  - Correlation (time vs accuracy): {correlation:.3f}")
if correlation < -0.1:
    print(f"  - Finding: Longer time = lower accuracy (struggling)")
elif correlation > 0.1:
    print(f"  - Finding: Longer time = higher accuracy (careful thinking)")
else:
    print(f"  - Finding: Weak correlation (time not predictive)")

ACCURACY BY TIME SPENT
             accuracy  count
time_bucket                 
0-5s         0.988011   2252
5-10s        0.702678   4332
10-15s       0.192927   2234
15s+         0.052453   1182

📊 Insight:
  - Correlation (time vs accuracy): -0.696
  - Finding: Longer time = lower accuracy (struggling)


---
## Query 5: User Performance Distribution

**Question:** How are users distributed by overall accuracy?

In [9]:
# Calculate accuracy per user
user_performance = df.groupby('user_id').agg({
    'is_correct': ['mean', 'count'],
    'user_level': 'first'
})

user_performance.columns = ['accuracy', 'total_attempts', 'level']

# Create performance categories
user_performance['category'] = pd.cut(user_performance['accuracy'],
                                       bins=[0, 0.4, 0.6, 0.8, 1.0],
                                       labels=['Struggling', 'Below Average', 'Good', 'Excellent'])

category_dist = user_performance['category'].value_counts().sort_index()

print("USER PERFORMANCE DISTRIBUTION")
print("=" * 50)
print(f"Total users: {len(user_performance)}\n")
for cat in category_dist.index:
    count = category_dist[cat]
    pct = count / len(user_performance) * 100
    print(f"{cat:15s}: {count:3d} users ({pct:5.1f}%)")

print("\n📊 Insight:")
print(f"  - Average user accuracy: {user_performance['accuracy'].mean():.1%}")
print(f"  - Median user accuracy: {user_performance['accuracy'].median():.1%}")
print(f"  - Users needing help (<40%): {(user_performance['accuracy'] < 0.4).sum()}")

USER PERFORMANCE DISTRIBUTION
Total users: 100

Struggling     :  29 users ( 29.0%)
Below Average  :  21 users ( 21.0%)
Good           :  39 users ( 39.0%)
Excellent      :  11 users ( 11.0%)

📊 Insight:
  - Average user accuracy: 57.7%
  - Median user accuracy: 59.5%
  - Users needing help (<40%): 28


---
## Query 6: Retention Analysis ⭐

**Question:** How does accuracy decay over time since last review?

**This is the data for the Sweet Spot chart!**

In [10]:
# Filter: only repeated attempts (attempt_number > 1)
retention_data = df[df['attempt_number'] > 1].copy()

# Group by days_since_last_seen
retention_curve = retention_data.groupby('days_since_last_seen').agg({
    'is_correct': ['mean', 'count']
})

retention_curve.columns = ['accuracy', 'count']

# Only include days with at least 10 samples
retention_curve = retention_curve[retention_curve['count'] >= 10]

print("RETENTION CURVE (Accuracy by Days Since Last Seen)")
print("=" * 60)
print(retention_curve.head(15).to_string())

# Find the "sweet spot" - day where accuracy drops significantly
# Define significant drop as > 10% from day 0-3 average
baseline_accuracy = retention_curve[retention_curve.index <= 3]['accuracy'].mean()
threshold = baseline_accuracy - 0.10

sweet_spot = retention_curve[retention_curve['accuracy'] < threshold].index.min()

print("\n" + "=" * 60)
print("📊 KEY INSIGHTS - SWEET SPOT OF REVIEW")
print("=" * 60)
print(f"Baseline accuracy (0-3 days): {baseline_accuracy:.1%}")
if pd.notna(sweet_spot):
    print(f"⭐ SWEET SPOT: Review before day {sweet_spot}")
    print(f"   Accuracy at day {sweet_spot}: {retention_curve.loc[sweet_spot, 'accuracy']:.1%}")
    print(f"   Drop from baseline: {baseline_accuracy - retention_curve.loc[sweet_spot, 'accuracy']:.1%}")
else:
    print("No significant drop detected in available data")

# Show accuracy at key intervals
print("\nAccuracy at key intervals:")
for day in [1, 3, 7, 14, 21, 30]:
    if day in retention_curve.index:
        acc = retention_curve.loc[day, 'accuracy']
        print(f"  Day {day:2d}: {acc:.1%}")

RETENTION CURVE (Accuracy by Days Since Last Seen)
                      accuracy  count
days_since_last_seen                 
1                     0.603922    765
2                     0.631242    781
3                     0.586634    808
5                     0.557522    339
6                     0.559603    302
7                     0.539185    319
10                    0.500000     76
11                    0.590361     83
12                    0.494253     87
13                    0.487179     78
14                    0.466667     75
15                    0.583333     12
16                    0.466667     15
17                    0.535714     28
18                    0.352941     17

📊 KEY INSIGHTS - SWEET SPOT OF REVIEW
Baseline accuracy (0-3 days): 60.7%
⭐ SWEET SPOT: Review before day 10
   Accuracy at day 10: 50.0%
   Drop from baseline: 10.7%

Accuracy at key intervals:
  Day  1: 60.4%
  Day  3: 58.7%
  Day  7: 53.9%
  Day 14: 46.7%
  Day 21: 52.6%
  Day 30: 40.0%


---
## Summary: Export Key Metrics

Save retention curve data for visualization in next step

In [11]:
# Export retention curve for Sweet Spot visualization
retention_curve.to_csv('../processed/retention_curve.csv', sep=';', encoding='utf-8-sig')
print("✅ Saved retention_curve.csv for visualization")

# Export summary statistics
summary_stats = {
    'total_results': len(df),
    'total_users': df['user_id'].nunique(),
    'total_words': df['word_id'].nunique(),
    'overall_accuracy': df['is_correct'].mean(),
    'avg_time_spent': df['time_spent'].mean(),
    'sweet_spot_day': sweet_spot if pd.notna(sweet_spot) else None,
    'baseline_accuracy': baseline_accuracy
}

pd.Series(summary_stats).to_csv('../processed/summary_stats.csv', sep=';', encoding='utf-8-sig')
print("✅ Saved summary_stats.csv")

print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)
print("Next step: Create visualizations (Sweet Spot chart!)")

✅ Saved retention_curve.csv for visualization
✅ Saved summary_stats.csv

ANALYSIS COMPLETE
Next step: Create visualizations (Sweet Spot chart!)
